# Evaluate Division

You are given an array of variable pairs equations and an array of real numbers `values`, where $equations[i] = [A_i, B_i]$ and `values[i]` represent the equation $A_{i} / B_{i} = values[i]$. Each Ai or Bi is a string that represents a single variable.

You are also given some queries, where $queries[j] = [C_j, D_j]$ represents the jth query where you must find the answer for $C_j / D_j = ?$.

Return the answers to all queries. If a single answer cannot be determined, return -1.0.

Note: The input is always valid. You may assume that evaluating the queries will not result in division by zero and that there is no contradiction.

Note: The variables that do not occur in the list of equations are undefined, so the answer cannot be determined for them.

### Example 1
Input: equations = [["a","b"],["b","c"]], values = [2.0,3.0], queries = [["a","c"],["b","a"],["a","e"],["a","a"],["x","x"]]\
Output: [6.00000,0.50000,-1.00000,1.00000,-1.00000]\

Explanation:\
Given: a / b = 2.0, b / c = 3.0\
queries are: a / c = ?, b / a = ?, a / e = ?, a / a = ?, x / x = ?\ 
return: [6.0, 0.5, -1.0, 1.0, -1.0 ]\
note: x is undefined => -1.0

### Example 2
Input: equations = [["a","b"],["b","c"],["bc","cd"]], values = [1.5,2.5,5.0], queries = [["a","c"],["c","b"],["bc","cd"],["cd","bc"]]\
Output: [3.75000,0.40000,5.00000,0.20000]

### Constraints
* 1 <= equations.length <= 20
* equations[i].length == 2
* 1 <= Ai.length, Bi.length <= 5
* values.length == equations.length
* 0.0 < values[i] <= 20.0
* 1 <= queries.length <= 20
* queries[i].length == 2
* 1 <= Cj.length, Dj.length <= 5
* Ai, Bi, Cj, Dj consist of lower case English letters and digits.

# Approach #1: Adjacency List Graph with breadth-first search

We can represent the problem space as a directed graph where each variable is a node. The edge `A -> B` represents the equation `A/B`. We can build an adjacency list `known` from the known `values` for each $[A_i,B_i]$ in `equations` such that $ known[A_i][B_i]= values[i]$. We can also add the inverses such that $ known[B_i][A_i] = 1/values[i] $.

Next, we'll need to implement a search function to run against every `query` in `queries`. The search function can be based on breadth-first search. To find unknown values for queries, we look for known values for the nominator and denominator and chain the operations for the corresponding equations together. For example, If we know that $/frac{A}{B} = 1 and /frac{B}{C} = 2$ , then we can chain up those two operations to find that $/frac{A}{C} = /frac{A}{B} * /frac{B}{C} = 1*2 = 2 $. We'll initialize the search queue with `(nom, denom, current_value=float(1), remainder = set())`. `current_value` tracks the tenative value I have for this equation, but is meaningless until certain conditions are met. `remainder` tracks all the denominators I need to cancel. For each item in the search_queue, we have 5 possible cases...

Case 1, neither the current nominator nor denominator are in the adjacency list `known`. We know we hit a dead end, so we just return -1. We do not want to store this answer in `known` because it would prevent us from hitting Case 1 and knowning we hit a deadend. Plus, this is an O(1) check, so it wouldn't be more efficient anyways.

Case 2, current_nominator==current_denominator, so we just return 1. 

Case 3, known[current_nominator][current_denominator] already exists and there's no `remainder` denominators to cancel out. We can simply return known[current_nominator][current_denominator]. Optionally we can also store known[query_n][query_d] = known[current_nominator][current_denominator] as well as the inverse.

Case 4, known[current_nominator][current_denominator] already exists but we only need to cancel out a denominator that happens to equal the query_n nominator. In this case, we return the current_value. And optionally store this new known value and its inverse.

Case 5, we know after all these cases that the current_nominator and current_denominator are in the adjacency list and that we still have `remainder` denominators to cancel out. In this case, we'll do the actual breath first search and try to see if we can find an answer by chaining the equations. We'll search adjacent nodes for the current_denominator, and only consider edges we haven't seen for this search. We'll remove the current_denominator from our remainder, then add its adjacent nodes. Then we divide the current_value by the value of the edge corresponding to the adjacent node. And then we add these new items to our search query. 

If we exhaust all the items in our search list without finding an answer, we simply return -1.

In [ ]:
from collections import deque
from typing import List

def calcEquation(equations: List[List[str]], values: List[float], queries: List[List[str]]) -> List[float]:
    known = dict()
    for index in range(len(equations)):
        nom, denom = equations[index]
        value = values[index]
        if nom not in known:
            known[nom] = dict()
        if denom not in known:
            known[denom] = dict()
        known[nom][denom] = value
        known[denom][nom] = 1/value

    answer = []

    def search_answer(nom, denom):
        nonlocal known
        seen = set([(nom,denom)])
        search_queue = deque([(nom,denom, float(1), set())])
        
        while search_queue:
            curr_nom, curr_denom, curr_val, remainder = search_queue.popleft()
            if known.get(curr_nom) == None or known.get(curr_denom) == None:
                return -1
            if curr_nom == curr_denom:
                return 1
            if known.get(curr_nom) and known[curr_nom].get(curr_denom) and not remainder:
                return known[curr_nom][curr_denom]
            if known.get(curr_nom) and known[curr_nom].get(curr_denom) and remainder == {nom}:
                return curr_val

            for adj_denom,adj_val in known[curr_denom].items():
                if (curr_denom, adj_denom) not in seen:
                    new_remainder = (remainder | {adj_denom}) - {curr_denom}
                    search_queue.append((curr_denom, adj_denom, (curr_val/adj_val), new_remainder))
                    seen.add((curr_denom, adj_denom))

        return -1

    for nom,denom in queries:
        answer.append(search_answer(nom,denom))
    
    return answer

# 